## YFinance Ticker Information Extraction

This notebook demonstrates how to:
- Extract ticker information using the `yfinance` library
- Get company profiles, financials, and fundamental data
- Retrieve investor relations URLs and corporate information
- Enrich the Dow 30 ticker data with comprehensive financial information


### Import Libraries


In [1]:
import yfinance as yf
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional
import time
from concurrent.futures import ThreadPoolExecutor, as_completed


### Configuration


In [2]:
# Directory paths
CATALOGUE_DIR = Path("../data/catalogue")
OUTPUT_DIR = Path("../data/structured")

# Create directories if they don't exist
CATALOGUE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DOW 30 Tickers (as of 2024)
DOW30_TICKERS = [
    'AAPL', 'AMGN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'DOW',
    'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM',
    'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'V', 'VZ', 'WBA', 'WMT'
]

print(f"✓ Configuration loaded")
print(f"  Catalogue directory: {CATALOGUE_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Total tickers: {len(DOW30_TICKERS)}")


✓ Configuration loaded
  Catalogue directory: ../data/catalogue
  Output directory: ../data/structured
  Total tickers: 30


### Extract Ticker Information using YFinance


In [3]:
def get_ticker_info(ticker: str, max_retries: int = 3) -> Dict[str, any]:
    """
    Get comprehensive ticker information from YFinance
    
    Args:
        ticker: Stock ticker symbol
        max_retries: Maximum number of retry attempts
        
    Returns:
        Dictionary containing ticker information
    """
    for attempt in range(max_retries):
        try:
            # Create ticker object
            stock = yf.Ticker(ticker)
            
            # Get the info dictionary
            info = stock.info
            
            # Extract key information
            ticker_data = {
                'ticker': ticker,
                'company_name': info.get('longName', info.get('shortName', ticker)),
                'sector': info.get('sector', None),
                'industry': info.get('industry', None),
                'website': info.get('website', None),
                'business_summary': info.get('longBusinessSummary', None),
                'headquarters_city': info.get('city', None),
                'headquarters_state': info.get('state', None),
                'headquarters_country': info.get('country', None),
                'phone': info.get('phone', None),
                'full_time_employees': info.get('fullTimeEmployees', None),
                
                # Market data
                'market_cap': info.get('marketCap', None),
                'current_price': info.get('currentPrice', info.get('regularMarketPrice', None)),
                'currency': info.get('currency', 'USD'),
                'exchange': info.get('exchange', None),
                
                # Financial metrics
                'pe_ratio': info.get('trailingPE', None),
                'forward_pe': info.get('forwardPE', None),
                'dividend_yield': info.get('dividendYield', None),
                'beta': info.get('beta', None),
                
                # Additional URLs
                'ir_website': info.get('irWebsite', None),
                'investor_relations_url': None,  # Will be constructed
                
                # Metadata
                'data_source': 'yfinance',
                'extraction_timestamp': datetime.now().isoformat(),
            }
            
            # Construct investor relations URL if not available
            if ticker_data['website'] and not ticker_data['ir_website']:
                base_website = ticker_data['website'].rstrip('/')
                # Common IR URL patterns
                ticker_data['investor_relations_url'] = f"{base_website}/investors"
            elif ticker_data['ir_website']:
                ticker_data['investor_relations_url'] = ticker_data['ir_website']
            
            return ticker_data
            
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  ⚠️  Retry {attempt + 1}/{max_retries} for {ticker}: {str(e)}")
                time.sleep(1)  # Wait before retry
            else:
                print(f"  ❌ Failed to get info for {ticker}: {str(e)}")
                return {
                    'ticker': ticker,
                    'company_name': None,
                    'error': str(e),
                    'data_source': 'yfinance',
                    'extraction_timestamp': datetime.now().isoformat(),
                }
    
    return None


# Test with a single ticker
print("Testing with AAPL...")
test_data = get_ticker_info('AAPL')
print(f"✓ Company: {test_data.get('company_name')}")
print(f"✓ Website: {test_data.get('website')}")
print(f"✓ Sector: {test_data.get('sector')}")
print(f"✓ Market Cap: ${test_data.get('market_cap', 0):,}" if test_data.get('market_cap') else "✓ Market Cap: N/A")


Testing with AAPL...
✓ Company: Apple Inc.
✓ Website: https://www.apple.com
✓ Sector: Technology
✓ Market Cap: $3,804,630,614,016


### Extract All Dow 30 Tickers (Parallel Processing)


In [4]:
def extract_all_tickers(tickers: List[str], max_workers: int = 5) -> List[Dict[str, any]]:
    """
    Extract information for multiple tickers using parallel processing
    
    Args:
        tickers: List of ticker symbols
        max_workers: Maximum number of parallel workers
        
    Returns:
        List of ticker data dictionaries
    """
    results = []
    
    print(f"{'='*60}")
    print(f"Extracting data for {len(tickers)} tickers...")
    print(f"{'='*60}\n")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_ticker = {executor.submit(get_ticker_info, ticker): ticker for ticker in tickers}
        
        # Process completed tasks
        for i, future in enumerate(as_completed(future_to_ticker), 1):
            ticker = future_to_ticker[future]
            try:
                data = future.result()
                if data:
                    results.append(data)
                    status = "✓" if data.get('company_name') else "⚠️"
                    print(f"{status} [{i:2}/{len(tickers)}] {ticker:6} - {data.get('company_name', 'N/A')}")
            except Exception as e:
                print(f"❌ [{i:2}/{len(tickers)}] {ticker:6} - Error: {str(e)}")
    
    print(f"\n{'='*60}")
    print(f"Extraction complete: {len(results)}/{len(tickers)} tickers")
    print(f"{'='*60}\n")
    
    return results


# Extract all Dow 30 tickers
all_ticker_data = extract_all_tickers(DOW30_TICKERS, max_workers=5)


Extracting data for 30 tickers...

✓ [ 1/30] AAPL   - Apple Inc.
✓ [ 2/30] CAT    - Caterpillar Inc.
✓ [ 3/30] AXP    - American Express Company
✓ [ 4/30] BA     - The Boeing Company
✓ [ 5/30] AMGN   - Amgen Inc.
✓ [ 6/30] CRM    - Salesforce, Inc.
✓ [ 7/30] CSCO   - Cisco Systems, Inc.
✓ [ 8/30] DIS    - The Walt Disney Company
✓ [ 9/30] CVX    - Chevron Corporation
✓ [10/30] DOW    - Dow Inc.
✓ [11/30] HD     - The Home Depot, Inc.
✓ [12/30] GS     - The Goldman Sachs Group, Inc.
✓ [13/30] IBM    - International Business Machines Corporation
✓ [14/30] HON    - Honeywell International Inc.
✓ [15/30] INTC   - Intel Corporation
✓ [16/30] JNJ    - Johnson & Johnson
✓ [17/30] JPM    - JPMorgan Chase & Co.
✓ [18/30] KO     - The Coca-Cola Company
✓ [19/30] MCD    - McDonald's Corporation
✓ [20/30] MMM    - 3M Company
✓ [21/30] MRK    - Merck & Co., Inc.
✓ [22/30] MSFT   - Microsoft Corporation
✓ [23/30] NKE    - NIKE, Inc.
✓ [24/30] PG     - The Procter & Gamble Company
✓ [25/30] UNH    - 

### Create DataFrame and Analyze


In [5]:
# Create DataFrame
df = pd.DataFrame(all_ticker_data)

# Sort by ticker
df = df.sort_values('ticker').reset_index(drop=True)

print(f"{'='*60}")
print("DATA SUMMARY")
print(f"{'='*60}")
print(f"Total companies: {len(df)}")
print(f"Companies with website: {df['website'].notna().sum()}")
print(f"Companies with IR URL: {df['investor_relations_url'].notna().sum()}")
print(f"Companies with business summary: {df['business_summary'].notna().sum()}")
print(f"{'='*60}\n")

# Display basic info
display_cols = ['ticker', 'company_name', 'website', 'sector', 'industry']
print("Basic Company Information:")
print(df[display_cols].to_string(index=False))


DATA SUMMARY
Total companies: 30
Companies with website: 30
Companies with IR URL: 30
Companies with business summary: 30

Basic Company Information:
ticker                                company_name                                website                 sector                            industry
  AAPL                                  Apple Inc.                  https://www.apple.com             Technology                Consumer Electronics
  AMGN                                  Amgen Inc.                  https://www.amgen.com             Healthcare        Drug Manufacturers - General
   AXP                    American Express Company        https://www.americanexpress.com     Financial Services                     Credit Services
    BA                          The Boeing Company                 https://www.boeing.com            Industrials                 Aerospace & Defense
   CAT                            Caterpillar Inc.            https://www.caterpillar.com            Indu

### Display Investor Relations URLs


In [6]:
print(f"{'='*80}")
print("INVESTOR RELATIONS URLs")
print(f"{'='*80}\n")

ir_cols = ['ticker', 'company_name', 'website', 'investor_relations_url']
df_ir = df[ir_cols].copy()

# Display companies with IR URLs
print("Companies with Investor Relations URLs:")
for idx, row in df_ir.iterrows():
    if pd.notna(row['investor_relations_url']):
        print(f"  {row['ticker']:6} - {row['company_name']:45} | {row['investor_relations_url']}")

print(f"\n{'='*80}")
print(f"Coverage: {df['investor_relations_url'].notna().sum()}/{len(df)} companies have IR URLs")
print(f"{'='*80}")


INVESTOR RELATIONS URLs

Companies with Investor Relations URLs:
  AAPL   - Apple Inc.                                    | http://investor.apple.com/
  AMGN   - Amgen Inc.                                    | http://investors.amgen.com/phoenix.zhtml?c=61656&p=irol-IRHome
  AXP    - American Express Company                      | http://ir.americanexpress.com/phoenix.zhtml?c=64467&p=irol-irhome
  BA     - The Boeing Company                            | http://www.boeing.com/boeing/companyoffices/financial/index.page?
  CAT    - Caterpillar Inc.                              | http://www.caterpillar.com/investors
  CRM    - Salesforce, Inc.                              | http://www.salesforce.com/company/investor/
  CSCO   - Cisco Systems, Inc.                           | https://www.cisco.com/investors
  CVX    - Chevron Corporation                           | http://investor.chevron.com/phoenix.zhtml?c=130102&p=irol-irhome
  DIS    - The Walt Disney Company                       | http

### Sector and Industry Analysis


In [7]:
print(f"{'='*60}")
print("SECTOR DISTRIBUTION")
print(f"{'='*60}")
sector_counts = df['sector'].value_counts()
for sector, count in sector_counts.items():
    print(f"  {sector:30} : {count:2} companies")

print(f"\n{'='*60}")
print("TOP INDUSTRIES")
print(f"{'='*60}")
industry_counts = df['industry'].value_counts().head(10)
for industry, count in industry_counts.items():
    print(f"  {industry:40} : {count:2} companies")


SECTOR DISTRIBUTION
  Technology                     :  6 companies
  Healthcare                     :  5 companies
  Financial Services             :  5 companies
  Industrials                    :  4 companies
  Consumer Cyclical              :  3 companies
  Consumer Defensive             :  3 companies
  Communication Services         :  2 companies
  Energy                         :  1 companies
  Basic Materials                :  1 companies

TOP INDUSTRIES
  Drug Manufacturers - General             :  3 companies
  Credit Services                          :  2 companies
  Conglomerates                            :  2 companies
  Consumer Electronics                     :  1 companies
  Banks - Diversified                      :  1 companies
  Pharmaceutical Retailers                 :  1 companies
  Telecom Services                         :  1 companies
  Healthcare Plans                         :  1 companies
  Insurance - Property & Casualty          :  1 companies
  Househol

### Save Output Files


In [8]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save full dataset
full_csv = OUTPUT_DIR / f"dow30_full_info_{timestamp}.csv"
full_json = OUTPUT_DIR / f"dow30_full_info_{timestamp}.json"

df.to_csv(full_csv, index=False)
df.to_json(full_json, orient='records', indent=2)

print(f"✓ Saved full dataset:")
print(f"  CSV : {full_csv}")
print(f"  JSON: {full_json}")

# Save basic catalog (ticker, name, website, IR URL)
catalog_cols = ['ticker', 'company_name', 'website', 'investor_relations_url', 'sector', 'industry']
df_catalog = df[catalog_cols].copy()

catalog_csv = CATALOGUE_DIR / f"dow30_catalog_{timestamp}.csv"
catalog_json = CATALOGUE_DIR / f"dow30_catalog_{timestamp}.json"

df_catalog.to_csv(catalog_csv, index=False)
df_catalog.to_json(catalog_json, orient='records', indent=2)

print(f"\n✓ Saved catalog:")
print(f"  CSV : {catalog_csv}")
print(f"  JSON: {catalog_json}")

# Save latest versions (without timestamp)
df.to_csv(OUTPUT_DIR / "dow30_full_info_latest.csv", index=False)
df.to_json(OUTPUT_DIR / "dow30_full_info_latest.json", orient='records', indent=2)

df_catalog.to_csv(CATALOGUE_DIR / "dow30_catalog_latest.csv", index=False)
df_catalog.to_json(CATALOGUE_DIR / "dow30_catalog_latest.json", orient='records', indent=2)

print(f"\n✓ Updated latest versions")


✓ Saved full dataset:
  CSV : ../data/structured/dow30_full_info_20251006_145648.csv
  JSON: ../data/structured/dow30_full_info_20251006_145648.json

✓ Saved catalog:
  CSV : ../data/catalogue/dow30_catalog_20251006_145648.csv
  JSON: ../data/catalogue/dow30_catalog_20251006_145648.json

✓ Updated latest versions


### Advanced: Get Financial Data (Optional)


In [9]:
def get_financial_statements(ticker: str) -> Dict[str, pd.DataFrame]:
    """
    Get financial statements for a ticker
    
    Returns:
        Dictionary with income_statement, balance_sheet, cash_flow
    """
    try:
        stock = yf.Ticker(ticker)
        
        return {
            'income_statement': stock.financials,
            'balance_sheet': stock.balance_sheet,
            'cash_flow': stock.cashflow,
            'quarterly_financials': stock.quarterly_financials
        }
    except Exception as e:
        print(f"Error getting financials for {ticker}: {e}")
        return {}


# Example: Get financials for Apple
print("Example: Getting financial statements for AAPL...")
aapl_financials = get_financial_statements('AAPL')

if aapl_financials.get('income_statement') is not None:
    print(f"\n✓ Income Statement shape: {aapl_financials['income_statement'].shape}")
    print(f"✓ Balance Sheet shape: {aapl_financials['balance_sheet'].shape}")
    print(f"✓ Cash Flow shape: {aapl_financials['cash_flow'].shape}")
    
    print("\nRecent Income Statement (first 5 rows):")
    print(aapl_financials['income_statement'].head())


Example: Getting financial statements for AAPL...

✓ Income Statement shape: (39, 4)
✓ Balance Sheet shape: (68, 4)
✓ Cash Flow shape: (53, 4)

Recent Income Statement (first 5 rows):
                                                      2024-09-30  \
Tax Effect Of Unusual Items                         0.000000e+00   
Tax Rate For Calcs                                  2.410000e-01   
Normalized EBITDA                                   1.346610e+11   
Net Income From Continuing Operation Net Minori...  9.373600e+10   
Reconciled Depreciation                             1.144500e+10   

                                                      2023-09-30  \
Tax Effect Of Unusual Items                         0.000000e+00   
Tax Rate For Calcs                                  1.470000e-01   
Normalized EBITDA                                   1.258200e+11   
Net Income From Continuing Operation Net Minori...  9.699500e+10   
Reconciled Depreciation                             1.151900e+10   

### Summary & Next Steps


In [10]:
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"✓ Extracted data for {len(df)} Dow 30 companies")
print(f"✓ Data fields collected:")
print(f"  - Company information (name, sector, industry)")
print(f"  - Website and Investor Relations URLs")
print(f"  - Headquarters location")
print(f"  - Market data (price, market cap, exchange)")
print(f"  - Financial metrics (P/E ratio, dividend yield, beta)")
print(f"  - Business summaries")
print(f"\n✓ Output files saved to:")
print(f"  - Full data: {OUTPUT_DIR}")
print(f"  - Catalog: {CATALOGUE_DIR}")
print(f"\n{'='*80}")
print("NEXT STEPS")
print(f"{'='*80}")
print("1. Validate investor relations URLs")
print("2. Scrape investor relations pages for annual reports")
print("3. Download PDF reports")
print("4. Parse and extract financial data from reports")
print(f"{'='*80}\n")

# Display final data quality metrics
print("DATA QUALITY METRICS:")
print(f"  Companies with websites: {df['website'].notna().sum()}/{len(df)} ({df['website'].notna().sum()/len(df)*100:.1f}%)")
print(f"  Companies with IR URLs: {df['investor_relations_url'].notna().sum()}/{len(df)} ({df['investor_relations_url'].notna().sum()/len(df)*100:.1f}%)")
print(f"  Companies with sector info: {df['sector'].notna().sum()}/{len(df)} ({df['sector'].notna().sum()/len(df)*100:.1f}%)")
print(f"  Companies with business summary: {df['business_summary'].notna().sum()}/{len(df)} ({df['business_summary'].notna().sum()/len(df)*100:.1f}%)")



SUMMARY
✓ Extracted data for 30 Dow 30 companies
✓ Data fields collected:
  - Company information (name, sector, industry)
  - Website and Investor Relations URLs
  - Headquarters location
  - Market data (price, market cap, exchange)
  - Financial metrics (P/E ratio, dividend yield, beta)
  - Business summaries

✓ Output files saved to:
  - Full data: ../data/structured
  - Catalog: ../data/catalogue

NEXT STEPS
1. Validate investor relations URLs
2. Scrape investor relations pages for annual reports
3. Download PDF reports
4. Parse and extract financial data from reports

DATA QUALITY METRICS:
  Companies with websites: 30/30 (100.0%)
  Companies with IR URLs: 30/30 (100.0%)
  Companies with sector info: 30/30 (100.0%)
  Companies with business summary: 30/30 (100.0%)
